# Weighted RRF search: LSTM + XGBoost

Owner: Chua Rui Ning, Renee

This notebook compares a small, explicit grid of weighted Reciprocal Rank Fusion (RRF) ratios on the same validation requests. It reports the best LSTM:XGBoost ratio separately for Recall@5/100/200 and observed-click nDCG@5/100/200.

It writes the fused top-200 Parquet for every ratio plus CSV summaries under Evaluation/ensemble/weights. The metrics follow the shared evaluator's macro-average definitions. This is validation model selection; freeze the selected ratio before the single final test evaluation.

Before running: verify both input exports cover all validation requests and were generated with the same catalogue and eligibility policy. A top-200 list alone cannot prove catalogue identity.

In [ ]:
from pathlib import Path
import ast
import json
import math
import sys

import numpy as np
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Update these paths if the experiment folders move.
PROJECT_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder')
VALIDATION_LABELS_PATH = PROJECT_ROOT / 'feature_engineering' / 'validation_features.parquet'

LSTM_RANKING_PATH = (
    PROJECT_ROOT / 'Experiments' / '2.1_lstm' / '2.1.3_improved_run'
    / '20261104_175920_improved_seed42' / 'rankings' / 'validation'
    / 'submission_top_200.parquet'
)
XGBOOST_RANKING_PATH = (
    PROJECT_ROOT / 'Experiments' / '1.3_xgboost' / '20261105_small_seed42'
    / 'rankings' / 'validation' / 'submission_top_200.parquet'
)

OUTPUT_DIR = PROJECT_ROOT / 'Evaluation' / 'ensemble' / 'weights'
TOP_K = 200
RRF_RANK_CONSTANT = 60

# LSTM share; XGBoost receives 1 - LSTM_WEIGHT.
# Includes the unweighted 50:50 baseline and model-only endpoints.
LSTM_WEIGHT_GRID = [1.00, 0.90, 0.75, 0.50, 0.25, 0.10, 0.00]

In [ ]:
EVALUATION_CODE_DIR = PROJECT_ROOT / 'Evaluation' / 'code'
if not (EVALUATION_CODE_DIR / 'evaluate.py').is_file():
    raise FileNotFoundError(
        f'Shared evaluator not found at {EVALUATION_CODE_DIR / "evaluate.py"}. '
        'Copy the Evaluation/code folder into the Drive project folder.'
    )
sys.path.insert(0, str(EVALUATION_CODE_DIR))
from evaluate import load_labels

required_paths = [VALIDATION_LABELS_PATH, LSTM_RANKING_PATH, XGBOOST_RANKING_PATH]
missing = [str(path) for path in required_paths if not Path(path).is_file()]
if missing:
    raise FileNotFoundError('Missing required inputs:\n' + '\n'.join(missing))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
request_frame, click_map = load_labels(VALIDATION_LABELS_PATH)
expected_ids = set(request_frame['behaviour_id'].astype(str))
request_order = request_frame['behaviour_id'].astype(str).tolist()
print(f'Validation requests: {len(expected_ids):,}')
print(f'Requests with known clicks: {sum(bool(click_map[x]) for x in request_order):,}')
print('Output folder:', OUTPUT_DIR)

Validation requests: 29,110
Requests with known clicks: 29,110
Output folder: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/ensemble/weights


In [ ]:
def _parse_article_ids(value, *, model_name, behaviour_id):
    if isinstance(value, str):
        try:
            value = json.loads(value)
        except json.JSONDecodeError:
            value = ast.literal_eval(value)
    if isinstance(value, np.ndarray):
        value = value.tolist()
    if not isinstance(value, (list, tuple)):
        raise TypeError(f'{model_name} {behaviour_id}: article_ids must be a list')
    ids = []
    for item in value:
        if pd.isna(item):
            raise ValueError(f'{model_name} {behaviour_id}: null article ID')
        number = int(item)
        if number != item:
            raise ValueError(f'{model_name} {behaviour_id}: non-integer article ID {item!r}')
        ids.append(number)
    if len(ids) < TOP_K:
        raise ValueError(f'{model_name} {behaviour_id}: expected at least {TOP_K} ranked articles, got {len(ids)}')
    ids = ids[:TOP_K]
    if len(set(ids)) != len(ids):
        raise ValueError(f'{model_name} {behaviour_id}: duplicate article IDs in top {TOP_K}')
    return ids


def load_rankings(path, model_name):
    path = Path(path)
    if path.suffix.lower() == '.csv':
        frame = pd.read_csv(path)
    elif path.suffix.lower() in {'.parquet', '.pq'}:
        frame = pd.read_parquet(path)
    else:
        raise ValueError(f'Unsupported ranking file: {path}')
    required = {'behaviour_id', 'article_ids'}
    if not required.issubset(frame.columns):
        raise ValueError(f'{path} must contain columns {sorted(required)}')
    if frame['behaviour_id'].isna().any():
        raise ValueError(f'{model_name}: null behaviour_id')
    frame['behaviour_id'] = frame['behaviour_id'].astype(str)
    if frame['behaviour_id'].duplicated().any():
        raise ValueError(f'{model_name}: duplicate behaviour_id rows')
    received = set(frame['behaviour_id'])
    if received != expected_ids:
        raise ValueError(
            f'{model_name} request coverage differs: '
            f'{len(expected_ids - received)} missing, {len(received - expected_ids)} unexpected'
        )
    ranking_map = {
        str(row.behaviour_id): _parse_article_ids(
            row.article_ids, model_name=model_name, behaviour_id=row.behaviour_id
        )
        for row in frame.itertuples(index=False)
    }
    print(f'{model_name}: validated {len(ranking_map):,} full-validation top-{TOP_K} lists')
    return ranking_map


lstm_rankings = load_rankings(LSTM_RANKING_PATH, 'LSTM')
xgboost_rankings = load_rankings(XGBOOST_RANKING_PATH, 'XGBoost')

LSTM: validated 29,110 full-validation top-200 lists
XGBoost: validated 29,110 full-validation top-200 lists


In [ ]:
def ranking_metrics(ranked_ids, clicked_ids, k):
    """Match evaluate.py: per-request known-click recall and binary observed-click nDCG."""
    clicked_ids = set(clicked_ids)
    if not clicked_ids:
        return np.nan, np.nan

    top = ranked_ids[:k]
    hits = sum(article_id in clicked_ids for article_id in top)
    recall = hits / len(clicked_ids)

    dcg = sum(
        1.0 / math.log2(rank + 1)
        for rank, article_id in enumerate(top, start=1)
        if article_id in clicked_ids
    )
    ideal_hits = min(len(clicked_ids), k)
    ideal_dcg = sum(1.0 / math.log2(rank + 1) for rank in range(1, ideal_hits + 1))
    return recall, dcg / ideal_dcg if ideal_dcg else np.nan


def summarize_rankings(ranking_map, model_name):
    records = []
    for behaviour_id in request_order:
        clicked = click_map[behaviour_id]
        record = {'behaviour_id': behaviour_id, 'model': model_name}
        for k in (5, 100, 200):
            recall, ndcg = ranking_metrics(ranking_map[behaviour_id], clicked, k)
            record[f'known_click_recall@{k}'] = recall
            record[f'observed_click_ndcg@{k}'] = ndcg
        records.append(record)
    per_request = pd.DataFrame.from_records(records)
    summary = {
        'model': model_name,
        'requests': int(len(per_request)),
        'requests_with_known_clicks': int(per_request['known_click_recall@5'].notna().sum()),
    }
    for column in per_request.columns:
        if column.startswith(('known_click_recall@', 'observed_click_ndcg@')):
            summary[column] = float(per_request[column].mean())
    return summary, per_request

In [ ]:
# Precompute each candidate's reciprocal-rank contribution once per request.
# Missing from a model's top 200 means no contribution from that model.
fusion_parts = {}
for behaviour_id in request_order:
    parts = {}
    for rank, article_id in enumerate(lstm_rankings[behaviour_id], start=1):
        parts.setdefault(article_id, [0.0, 0.0, rank])[0] = 1.0 / (RRF_RANK_CONSTANT + rank)
        parts[article_id][2] = min(parts[article_id][2], rank)
    for rank, article_id in enumerate(xgboost_rankings[behaviour_id], start=1):
        parts.setdefault(article_id, [0.0, 0.0, rank])[1] = 1.0 / (RRF_RANK_CONSTANT + rank)
        parts[article_id][2] = min(parts[article_id][2], rank)
    fusion_parts[behaviour_id] = parts


def fuse_weighted_rrf(lstm_weight):
    xgboost_weight = 1.0 - lstm_weight
    output_rows = []
    output_map = {}
    for request_index, behaviour_id in enumerate(request_order, start=1):
        parts = fusion_parts[behaviour_id]
        fused_ids = sorted(
            parts,
            key=lambda article_id: (
                -(lstm_weight * parts[article_id][0] + xgboost_weight * parts[article_id][1]),
                parts[article_id][2],
                article_id,
            ),
        )[:TOP_K]
        if len(fused_ids) != TOP_K:
            raise RuntimeError(f'{behaviour_id}: RRF produced fewer than {TOP_K} articles')
        output_map[behaviour_id] = fused_ids
        output_rows.append({'behaviour_id': behaviour_id, 'article_ids': fused_ids})
        if request_index % 5000 == 0:
            print(f'  fused {request_index:,}/{len(request_order):,} requests')
    return output_map, pd.DataFrame(output_rows)


# Sanity check: endpoints reproduce the source model rankings exactly.
endpoint_lstm, _ = fuse_weighted_rrf(1.0)
endpoint_xgb, _ = fuse_weighted_rrf(0.0)
assert all(endpoint_lstm[b] == lstm_rankings[b] for b in request_order)
assert all(endpoint_xgb[b] == xgboost_rankings[b] for b in request_order)
print('Endpoint checks passed: 100:0 reproduces LSTM; 0:100 reproduces XGBoost.')

  fused 5,000/29,110 requests
  fused 10,000/29,110 requests
  fused 15,000/29,110 requests
  fused 20,000/29,110 requests
  fused 25,000/29,110 requests
  fused 5,000/29,110 requests
  fused 10,000/29,110 requests
  fused 15,000/29,110 requests
  fused 20,000/29,110 requests
  fused 25,000/29,110 requests
Endpoint checks passed: 100:0 reproduces LSTM; 0:100 reproduces XGBoost.


In [ ]:
rows = []
metric_columns = [
    f'{metric}@{k}'
    for k in (5, 100, 200)
    for metric in ('known_click_recall', 'observed_click_ndcg')
]

for trial, lstm_weight in enumerate(LSTM_WEIGHT_GRID, start=1):
    xgboost_weight = 1.0 - lstm_weight
    ratio_name = f'lstm_{round(lstm_weight * 100):02d}_xgboost_{round(xgboost_weight * 100):02d}'
    print(f'[{trial}/{len(LSTM_WEIGHT_GRID)}] Evaluating {ratio_name}...')
    ranking_map, ranking_frame = fuse_weighted_rrf(lstm_weight)
    summary, _ = summarize_rankings(ranking_map, ratio_name)
    summary['lstm_weight'] = float(lstm_weight)
    summary['xgboost_weight'] = float(xgboost_weight)
    summary['ratio'] = f'{round(lstm_weight * 100)}:{round(xgboost_weight * 100)}'
    rows.append(summary)

    output_path = OUTPUT_DIR / f'weighted_rrf_{ratio_name}_validation_top_{TOP_K}.parquet'
    ranking_frame.to_parquet(output_path, index=False)
    print('  saved:', output_path.name)

results = pd.DataFrame(rows)
ordered_columns = ['ratio', 'lstm_weight', 'xgboost_weight', 'requests', 'requests_with_known_clicks'] + metric_columns
results = results[ordered_columns].sort_values('lstm_weight', ascending=False).reset_index(drop=True)
results_path = OUTPUT_DIR / 'weighted_rrf_validation_metrics.csv'
results.to_csv(results_path, index=False)

# Show a winner separately for every requested metric. On exact ties, prefer
# the ratio closest to the current 50:50 baseline.
winner_rows = []
for metric in metric_columns:
    best_value = results[metric].max()
    tied = results[np.isclose(results[metric], best_value, rtol=0, atol=1e-12)].copy()
    tied['distance_from_equal_weights'] = (tied['lstm_weight'] - 0.5).abs()
    winner = tied.sort_values('distance_from_equal_weights').iloc[0]
    winner_rows.append({
        'metric': metric,
        'best_ratio': winner['ratio'],
        'lstm_weight': winner['lstm_weight'],
        'xgboost_weight': winner['xgboost_weight'],
        'metric_value': winner[metric],
        'exact_tie_count': int(len(tied)),
    })

winners = pd.DataFrame(winner_rows)
winners_path = OUTPUT_DIR / 'weighted_rrf_best_weights_by_metric.csv'
winners.to_csv(winners_path, index=False)

print('\nAll weighted RRF validation results:')
display(results.round(6))
print('\nBest ratio by metric:')
display(winners.round(6))
print('\nSaved:')
print(results_path)
print(winners_path)

[1/7] Evaluating lstm_100_xgboost_00...
  fused 5,000/29,110 requests
  fused 10,000/29,110 requests
  fused 15,000/29,110 requests
  fused 20,000/29,110 requests
  fused 25,000/29,110 requests
  saved: weighted_rrf_lstm_100_xgboost_00_validation_top_200.parquet
[2/7] Evaluating lstm_90_xgboost_10...
  fused 5,000/29,110 requests
  fused 10,000/29,110 requests
  fused 15,000/29,110 requests
  fused 20,000/29,110 requests
  fused 25,000/29,110 requests
  saved: weighted_rrf_lstm_90_xgboost_10_validation_top_200.parquet
[3/7] Evaluating lstm_75_xgboost_25...
  fused 5,000/29,110 requests
  fused 10,000/29,110 requests
  fused 15,000/29,110 requests
  fused 20,000/29,110 requests
  fused 25,000/29,110 requests
  saved: weighted_rrf_lstm_75_xgboost_25_validation_top_200.parquet
[4/7] Evaluating lstm_50_xgboost_50...
  fused 5,000/29,110 requests
  fused 10,000/29,110 requests
  fused 15,000/29,110 requests
  fused 20,000/29,110 requests
  fused 25,000/29,110 requests
  saved: weighted_rrf_

,ratio,lstm_weight,xgboost_weight,requests,requests_with_known_clicks,known_click_recall@5,observed_click_ndcg@5,known_click_recall@100,observed_click_ndcg@100,known_click_recall@200,observed_click_ndcg@200
0,100:0,1.00,0.00,29110,29110,0.182542,0.109382,0.916103,0.286070,0.947656,0.290513
1,90:10,0.90,0.10,29110,29110,0.198472,0.122927,0.915966,0.295543,0.947656,0.300006
2,75:25,0.75,0.25,29110,29110,0.226086,0.143691,0.914283,0.307936,0.947656,0.312644
3,50:50,0.50,0.50,29110,29110,0.207834,0.133267,0.910916,0.294624,0.940202,0.298759
4,25:75,0.25,0.75,29110,29110,0.146305,0.096022,0.902019,0.259717,0.924932,0.262989
5,10:90,0.10,0.90,29110,29110,0.109193,0.070473,0.880838,0.234908,0.922665,0.240913
6,0:100,0.00,1.00,29110,29110,0.096848,0.059720,0.858227,0.221182,0.922665,0.230362



Best ratio by metric:


,metric,best_ratio,lstm_weight,xgboost_weight,metric_value,exact_tie_count
0,known_click_recall@5,75:25,0.75,0.25,0.226086,1
1,observed_click_ndcg@5,75:25,0.75,0.25,0.143691,1
2,known_click_recall@100,100:0,1.00,0.00,0.916103,1
3,observed_click_ndcg@100,75:25,0.75,0.25,0.307936,1
4,known_click_recall@200,75:25,0.75,0.25,0.947656,3
5,observed_click_ndcg@200,75:25,0.75,0.25,0.312644,1



Saved:
/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/ensemble/weights/weighted_rrf_validation_metrics.csv
/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/ensemble/weights/weighted_rrf_best_weights_by_metric.csv


## How to read the output

- weighted_rrf_validation_metrics.csv contains all tested ratios.
- weighted_rrf_best_weights_by_metric.csv names the best ratio separately for each of Recall@5/100/200 and nDCG@5/100/200.
- Each ratio's fused top-200 file is saved beside the CSVs. Use the selected ratio's Parquet as the reranker input if you choose to continue with it.
- Different metrics may prefer different ratios; there is no single best ratio unless the group predefines a primary metric or composite objective.
- Recall/nDCG here are macro-averaged per request with at least one known click, using the shared evaluator's definitions. Clicks absent from the logged labels cannot be evaluated.
- RRF weights are validation-tuned. Freeze the chosen ratio and all later reranker settings before evaluating test once. Do not select weights from test metrics.